In [14]:
# ============================================================
# Install Biopython
# ============================================================
!pip install biopython -q

from Bio import Entrez, SeqIO
import os, re, time
import pandas as pd

Entrez.email = "romens72@gmail.com"   # keep your email here

outdir = "validated_target_protein_sequences"
os.makedirs(outdir, exist_ok=True)

# ============================================================
# Target definitions with strict filters
# ============================================================

targets = {
    "Aspergillus_fumigatus_PksP_Alb1": {
        "organism": "Aspergillus fumigatus",
        "query": '("PksP" OR "Alb1" OR "conidial pigment polyketide synthase" OR "Afu2g17600" OR "AFUA_2G17600")',
        "must_include": ["polyketide synthase", "conidial pigment"],
        "min_len": 1500,
        "max_len": 3000
    },

    "Penicillium_oxalicum_CYP51": {
        "organism": "Penicillium oxalicum",
        "query": '("CYP51" OR "ERG11" OR "sterol 14-alpha demethylase" OR "lanosterol 14-alpha demethylase")',
        "must_include": ["cyp51", "erg11", "sterol 14-alpha", "lanosterol 14-alpha", "eburicol 14-alpha"],
        "min_len": 450,
        "max_len": 600
    },

    "Aspergillus_niger_CYP51": {
        "organism": "Aspergillus niger",
        "query": '("CYP51" OR "ERG11" OR "sterol 14-alpha demethylase" OR "lanosterol 14-alpha demethylase")',
        "must_include": ["cyp51", "erg11", "sterol 14-alpha", "lanosterol 14-alpha"],
        "min_len": 450,
        "max_len": 600
    }
}

def clean_filename(text):
    return re.sub(r"[^\w\-\.]", "_", text)[:120]

def search_ncbi(target_name, info, retmax=30):
    search_query = f'{info["query"]} AND "{info["organism"]}"[Organism]'

    print("\n" + "="*80)
    print("Target:", target_name)
    print("Search query:", search_query)

    handle = Entrez.esearch(
        db="protein",
        term=search_query,
        retmax=retmax
    )
    record = Entrez.read(handle)
    handle.close()

    ids = record["IdList"]
    print("Total candidates found:", len(ids))

    if not ids:
        return None, []

    # Fetch candidate FASTA records
    fetch_handle = Entrez.efetch(
        db="protein",
        id=",".join(ids),
        rettype="fasta",
        retmode="text"
    )

    records = list(SeqIO.parse(fetch_handle, "fasta"))
    fetch_handle.close()

    candidates = []

    for rec in records:
        desc = rec.description.lower()
        length = len(rec.seq)

        name_pass = any(term.lower() in desc for term in info["must_include"])
        len_pass = info["min_len"] <= length <= info["max_len"]

        candidates.append({
            "target": target_name,
            "id": rec.id,
            "description": rec.description,
            "length": length,
            "name_pass": name_pass,
            "length_pass": len_pass,
            "record": rec
        })

    # Prefer candidates passing both name and length filters
    valid = [c for c in candidates if c["name_pass"] and c["length_pass"]]

    if valid:
        selected = valid[0]
        print("Selected VALID sequence:")
    else:
        selected = None
        print("No fully valid sequence found. Review candidates manually.")

    # Print candidates
    for i, c in enumerate(candidates[:15], 1):
        print(f"\nCandidate {i}")
        print("ID:", c["id"])
        print("Length:", c["length"])
        print("Name pass:", c["name_pass"])
        print("Length pass:", c["length_pass"])
        print("Description:", c["description"])

    return selected, candidates

# ============================================================
# Run search and save only validated sequences
# ============================================================

summary = []
all_candidates = []

for target_name, info in targets.items():
    selected, candidates = search_ncbi(target_name, info)

    for c in candidates:
        all_candidates.append({
            "target": c["target"],
            "id": c["id"],
            "description": c["description"],
            "length": c["length"],
            "name_pass": c["name_pass"],
            "length_pass": c["length_pass"]
        })

    if selected:
        filename = clean_filename(target_name) + ".fasta"
        filepath = os.path.join(outdir, filename)

        SeqIO.write(selected["record"], filepath, "fasta")

        summary.append({
            "target": target_name,
            "selected_id": selected["id"],
            "description": selected["description"],
            "length": selected["length"],
            "file": filepath,
            "status": "saved"
        })

        print("\nSaved:", filepath)

    else:
        summary.append({
            "target": target_name,
            "selected_id": None,
            "description": None,
            "length": None,
            "file": None,
            "status": "manual review needed"
        })

    time.sleep(1)

# ============================================================
# Save candidate and summary tables
# ============================================================

summary_df = pd.DataFrame(summary)
candidates_df = pd.DataFrame(all_candidates)

summary_df.to_csv("selected_sequence_summary.csv", index=False)
candidates_df.to_csv("all_candidate_sequences.csv", index=False)

print("\n\nFINAL SUMMARY")
display(summary_df)

print("\nAll candidates saved to: all_candidate_sequences.csv")
print("Selected summary saved to: selected_sequence_summary.csv")


Target: Aspergillus_fumigatus_PksP_Alb1
Search query: ("PksP" OR "Alb1" OR "conidial pigment polyketide synthase" OR "Afu2g17600" OR "AFUA_2G17600") AND "Aspergillus fumigatus"[Organism]
Total candidates found: 30
Selected VALID sequence:

Candidate 1
ID: sp|B0XYE0.1|RLMA_ASPFC
Length: 600
Name pass: False
Length pass: False
Description: sp|B0XYE0.1|RLMA_ASPFC RecName: Full=MADS-box transcription factor rlmA

Candidate 2
ID: sp|B0XU60.1|DEVR_ASPFC
Length: 525
Name pass: False
Length pass: False
Description: sp|B0XU60.1|DEVR_ASPFC RecName: Full=Basic helix-loop-helix transcription factor devR

Candidate 3
ID: sp|Q4WX78.1|RLMA_ASPFU
Length: 600
Name pass: False
Length pass: False
Description: sp|Q4WX78.1|RLMA_ASPFU RecName: Full=Transcription factor rlmA

Candidate 4
ID: sp|Q4WRY5.1|LAEA_ASPFU
Length: 373
Name pass: False
Length pass: False
Description: sp|Q4WRY5.1|LAEA_ASPFU RecName: Full=Secondary metabolism regulator laeA; AltName: Full=Methyltransferase laeA; AltName: Full=Velvet co

,target,selected_id,description,length,file,status
0,Aspergillus_fumigatus_PksP_Alb1,XP_756095.1,XP_756095.1 polyketide synthase alb1 [Aspergil...,2146,validated_target_protein_sequences/Aspergillus...,saved
1,Penicillium_oxalicum_CYP51,KAI2785840.1,KAI2785840.1 Eburicol 14-alpha-demethylase [Pe...,525,validated_target_protein_sequences/Penicillium...,saved
2,Aspergillus_niger_CYP51,KAN0694988.1,KAN0694988.1 Lanosterol 14-alpha-demethylase [...,524,validated_target_protein_sequences/Aspergillus...,saved



All candidates saved to: all_candidate_sequences.csv
Selected summary saved to: selected_sequence_summary.csv


In [15]:
# Check saved FASTA files
!ls -lh validated_target_protein_sequences

# Zip FASTA files + summary CSVs
!zip -r validated_target_protein_sequences.zip validated_target_protein_sequences selected_sequence_summary.csv all_candidate_sequences.csv

# Download to your computer
from google.colab import files
files.download("validated_target_protein_sequences.zip")

total 12K
-rw-r--r-- 1 root root 2.2K Apr 27 07:53 Aspergillus_fumigatus_PksP_Alb1.fasta
-rw-r--r-- 1 root root  599 Apr 27 07:53 Aspergillus_niger_CYP51.fasta
-rw-r--r-- 1 root root  601 Apr 27 07:53 Penicillium_oxalicum_CYP51.fasta
  adding: validated_target_protein_sequences/ (stored 0%)
  adding: validated_target_protein_sequences/Aspergillus_niger_CYP51.fasta (deflated 31%)
  adding: validated_target_protein_sequences/Penicillium_oxalicum_CYP51.fasta (deflated 31%)
  adding: validated_target_protein_sequences/Aspergillus_fumigatus_PksP_Alb1.fasta (deflated 39%)
  adding: selected_sequence_summary.csv (deflated 55%)
  adding: all_candidate_sequences.csv (deflated 83%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>